# Combined decision record: eight existing projects

This notebook combines the three PyPI packages and five GitHub repositories. “Observed” means executed here; “source” means the implementation was inspected but could not be run as claimed; “documentation” is never silently promoted to a measurement.

In [1]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
pd.set_option("display.max_colwidth", 100)

projects = pd.DataFrame([
    ("swing-analyzer 0.1.2", "A", "wheel ran", "MediaPipe", "wrist heuristic", "none"),
    ("golftracker 2.0", "B", "capture/labels + adapter", "MediaPipe", "stored labels", "manual + interpolation"),
    ("golftrainer 1.0.2", "B", "table adapter only", "none", "none", "clubhead passthrough"),
    ("greggjuri/golf-swing-analyzer", "B", "video/events/club components", "placeholder", "whole-frame motion", "Hough; unreliable"),
    ("Heleena golf-swing-analysis", "A", "included video ran", "MediaPipe", "none", "none; fixed ball point"),
    ("Pose2Par", "C", "unreproducible checkout", "claimed MMPose", "wrist-Z heuristic", "none"),
    ("GolfPosePro", "B", "extraction + phase cells", "MediaPipe", "wrist-Y heuristic", "none"),
    ("Squat-AI", "C", "source + separate backend probe", "YOLO pose", "rep hysteresis", "none"),
], columns=["project", "level", "execution basis", "pose", "temporal logic", "club/ball"])
display(projects)

,project,level,execution basis,pose,temporal logic,club/ball
0,swing-analyzer 0.1.2,A,wheel ran,MediaPipe,wrist heuristic,none
1,golftracker 2.0,B,capture/labels + adapter,MediaPipe,stored labels,manual + interpolation
2,golftrainer 1.0.2,B,table adapter only,none,none,clubhead passthrough
3,greggjuri/golf-swing-analyzer,B,video/events/club components,placeholder,whole-frame motion,Hough; unreliable
4,Heleena golf-swing-analysis,A,included video ran,MediaPipe,none,none; fixed ball point
5,Pose2Par,C,unreproducible checkout,claimed MMPose,wrist-Z heuristic,none
6,GolfPosePro,B,extraction + phase cells,MediaPipe,wrist-Y heuristic,none
7,Squat-AI,C,source + separate backend probe,YOLO pose,rep hysteresis,none


In [2]:
mp = json.loads((ROOT / "outputs/pose_backends/00001_mediapipe_summary.json").read_text())
yolo = json.loads((ROOT / "outputs/pose_backends/00001_yolov8_summary.json").read_text())
pose = pd.DataFrame([
    ("MediaPipe Pose", mp["detected_frames"], mp["landmarks_per_detection"], mp["mean_latency_seconds"], "x/y/z/visibility; Apache-2.0 upstream"),
    ("YOLOv8n-pose (upgraded probe)", yolo["detected_frames"], yolo["landmarks_per_detection"], yolo["mean_latency_seconds"], "pixel x/y/confidence; Ultralytics licence review needed"),
    ("Pose2Par MMPose", np.nan, np.nan, np.nan, "not reproducible from clone; exact config unknown"),
], columns=["backend", "frames detected / 136", "points", "mean CPU seconds/frame", "raw output/licence note"])
display(pose)

,backend,frames detected / 136,points,mean CPU seconds/frame,raw output/licence note
0,MediaPipe Pose,136.0,33.0,0.026576,x/y/z/visibility; Apache-2.0 upstream
1,YOLOv8n-pose (upgraded probe),136.0,17.0,0.054481,pixel x/y/confidence; Ultralytics licence review needed
2,Pose2Par MMPose,NaN,NaN,NaN,not reproducible from clone; exact config unknown


In [3]:
events = pd.DataFrame([
    ("swing-analyzer", "observed", "wrist height/velocity", "address/top/impact/finish", "transparent baseline; no labelled validation"),
    ("golftracker", "observed stored data", "manual phase field", "free text/labels", "provenance useful; not automatic"),
    ("Greg", "observed", "whole-frame absdiff", "P1/P4/P7", "camera/global motion can dominate"),
    ("GolfPosePro", "observed", "mean wrist Y + velocity", "six ranges", "best executable normalization seed; unvalidated"),
    ("Pose2Par", "source only", "wrist Z", "address/top/impact/finish", "project validation claims not independently reproduced"),
], columns=["project", "evidence", "signal", "events", "decision"])
display(events)

club = pd.DataFrame([
    ("golftracker", "manual points + straight interpolation", "best audited labels/storage, not automatic tracking"),
    ("Greg", "Canny/Hough line + circle", "1/24 manual frames within 50 px for circle centre"),
    ("Heleena", "fixed ball pixel only", "drawing reference, not a detection"),
    ("others", "none", "no observed club trajectory"),
], columns=["project", "method", "finding"])
display(club)

,project,evidence,signal,events,decision
0,swing-analyzer,observed,wrist height/velocity,address/top/impact/finish,transparent baseline; no labelled validation
1,golftracker,observed stored data,manual phase field,free text/labels,provenance useful; not automatic
2,Greg,observed,whole-frame absdiff,P1/P4/P7,camera/global motion can dominate
3,GolfPosePro,observed,mean wrist Y + velocity,six ranges,best executable normalization seed; unvalidated
4,Pose2Par,source only,wrist Z,address/top/impact/finish,project validation claims not independently reproduced


,project,method,finding
0,golftracker,manual points + straight interpolation,"best audited labels/storage, not automatic tracking"
1,Greg,Canny/Hough line + circle,1/24 manual frames within 50 px for circle centre
2,Heleena,fixed ball pixel only,"drawing reference, not a detection"
3,others,none,no observed club trajectory


In [4]:
raw = pd.DataFrame([
    ("golftracker", "full 33-point MediaPipe row + visibility", "club provenance", "pickle/JSON schema is fragile"),
    ("MediaPipe probe", "long table: frame/time/name/x/y/z/visibility", "none", "best observed body-data starting point"),
    ("YOLO probe", "long table: frame/time/name/pixel x/y/confidence", "none", "17 points; version/licence caveat"),
    ("GolfPosePro", "average wrist Y only", "none", "too lossy"),
    ("Heleena", "no numeric export", "fixed ball drawing", "video only"),
    ("Pose2Par", "intended TRC/MOT", "none", "not reproduced"),
], columns=["source", "continuous body data", "club/ball", "assessment"])
display(raw)

,source,continuous body data,club/ball,assessment
0,golftracker,full 33-point MediaPipe row + visibility,club provenance,pickle/JSON schema is fragile
1,MediaPipe probe,long table: frame/time/name/x/y/z/visibility,none,best observed body-data starting point
2,YOLO probe,long table: frame/time/name/pixel x/y/confidence,none,17 points; version/licence caveat
3,GolfPosePro,average wrist Y only,none,too lossy
4,Heleena,no numeric export,fixed ball drawing,video only
5,Pose2Par,intended TRC/MOT,none,not reproduced


## Answers to the decision questions

1. **Which repositories cloned?** All five cloned successfully; exact commits are in `data/manifests/repository-manifest.yaml`.
2. **Which ran end to end?** Heleena only (Level A), on its included 465-frame video.
3. **Which ran partially?** Greg’s loader/events/club components and GolfPosePro’s exact extraction/phase functions (Level B).
4. **Which were source review only?** Pose2Par and Squat-AI (Level C). The modern YOLO output is only a separately labelled backend probe.
5. **Best pose approach observed?** MediaPipe: 33 landmarks, complete shared-clip coverage, CPU execution, visibility/Z, and permissive upstream licensing. Current YOLO was also complete but has 17 points and version/licensing friction; MMPose was not testable.
6. **Best event approach observed?** None is production-ready. GolfPosePro is the most useful executable phase-normalization seed; `swing-analyzer` is the clearest simple baseline. Both need labelled validation.
7. **Best club approach observed?** No automatic implementation passed scrutiny. Golftracker’s manual points and provenance are the best ground-truth asset; Greg remains a negative automatic baseline.
8. **Most useful continuous-data representation?** A long frame/timestamp table of every named landmark coordinate and confidence, plus clubhead/shaft/ball observations with source and confidence. Preserve raw values and missingness.
9. **What transfers from Squat-AI?** Confidence gating, explicit view/side assumptions, hysteresis, and GUI/inference separation. Its literal thresholds and geometry labels do not.
10. **What still needs development?** Versioned schemas; QC and annotation; validated events; calibrated geometry; evaluated club/ball observation; uncertainty-aware filtering; longitudinal variability/anomaly analysis; outcome joins; and independent validation.

In [5]:
status = pd.DataFrame([
    ("body landmarks", "observed_raw/model_estimate/low_confidence", "coordinate system + confidence + model version"),
    ("image angle", "derived_2d", "points, view, formula; never imply 3-D"),
    ("event", "heuristic_event/manual_label", "method, probability or interval, label provenance"),
    ("club/ball", "observed_raw/manual_label/interpolated/not_observable", "source per frame; gap limits"),
    ("OpenSim metric", "calibrated_kinematic", "model/calibration/IK residuals and any manual offset"),
], columns=["output", "allowed status", "minimum metadata"])
display(status)

projects.to_csv(ROOT / "outputs/repository_comparison.csv", index=False)
print("EXECUTED_OK: combined repository comparison notebook")

,output,allowed status,minimum metadata
0,body landmarks,observed_raw/model_estimate/low_confidence,coordinate system + confidence + model version
1,image angle,derived_2d,"points, view, formula; never imply 3-D"
2,event,heuristic_event/manual_label,"method, probability or interval, label provenance"
3,club/ball,observed_raw/manual_label/interpolated/not_observable,source per frame; gap limits
4,OpenSim metric,calibrated_kinematic,model/calibration/IK residuals and any manual offset


EXECUTED_OK: combined repository comparison notebook
